# 04 - Evaluation and Visualization

This notebook creates the required evaluation charts: confusion matrix heatmap, feature importance chart, and label distribution chart.

In [ ]:
from pathlib import Path
import sys
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.csp_features import FEATURE_COLUMNS

DATA_DIR = PROJECT_ROOT / 'data'
MODEL_DIR = PROJECT_ROOT / 'models'
FIGURE_DIR = PROJECT_ROOT / 'reports' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
df = pd.read_csv(DATA_DIR / 'csp_features_labeled.csv')
model = joblib.load(MODEL_DIR / 'csp_random_forest.pkl')

X = df[FEATURE_COLUMNS]
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
predictions = model.predict(X_test)

print('Accuracy:', accuracy_score(y_test, predictions))
print(classification_report(y_test, predictions))

In [ ]:
labels = ['Weak', 'Medium', 'Strong']
cm = confusion_matrix(y_test, predictions, labels=labels)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance, x='importance', y='feature', color='#2f6f8f')
plt.title('Feature Importance')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'feature_importance.png', dpi=150)
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='label', order=labels, palette=['#d9534f', '#f0ad4e', '#5cb85c'], hue='label', legend=False)
plt.title('Label Distribution')
plt.xlabel('CSP Security Label')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(FIGURE_DIR / 'label_distribution.png', dpi=150)
plt.show()